# Prep 1 · NumPy → JAX

**Time:** about an hour. **Needs:** `pixi install` done, nothing else.

The whole project is written in [JAX](https://docs.jax.dev/). If you can write NumPy you already know
90 % of it. This notebook covers the other 10 %: five habits that are different, and one bridge into the
repo — your first two milestones.

Run every cell, read every comment, and do the four exercises. Each exercise has a **check** cell:
when it runs without an `AssertionError`, you're done. Below each exercise sits a **collapsed hint** —
the struggle before you open it is where the learning happens, so give every exercise an honest
attempt (say, 15 minutes) before you click.

In [3]:
import numpy as np
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "· devices:", jax.devices())

JAX 0.11.1 · devices: [CpuDevice(id=0)]


## 1. `jnp` is `np`

Almost every NumPy function exists under `jax.numpy` with the same name and signature. Arrays convert
back and forth freely. Two differences to notice: JAX defaults to `float32` (fine for us — the images
are `float32` too), and a JAX array lives on whatever device JAX found (CPU here; the GPU at the school).

In [4]:
x_np = np.linspace(0, 1, 5)
x = jnp.asarray(x_np)          # NumPy -> JAX
print(type(x), x.dtype)
print(jnp.sin(x) ** 2 + jnp.cos(x) ** 2)   # same maths, same names
print(np.asarray(x))           # JAX -> NumPy (for matplotlib, scikit-image, ...)

<class 'jaxlib._jax.ArrayImpl'> float32
[1.         1.         1.         1.         0.99999994]
[0.   0.25 0.5  0.75 1.  ]


## 2. Arrays are immutable

In NumPy you write `x[0] = 1`. In JAX that raises an error: arrays are values, not buffers. Instead you
ask for a *new* array with the change applied: `x.at[0].set(1)`. This is what makes JAX able to
differentiate and compile your code — every operation is a pure function of its inputs.

In [5]:
x = jnp.zeros(4)
try:
    x[0] = 1.0
except TypeError as e:
    print("NumPy habit ->", type(e).__name__, ":", str(e)[:60], "...")

y = x.at[0].set(1.0)           # the JAX way: a new array
print("x is unchanged:", x)
print("y:", y)
print("whole columns at once:", jnp.zeros((3, 6)).at[:, ::2].set(1.0))

NumPy habit -> TypeError : JAX arrays are immutable and do not support in-place item as ...
x is unchanged: [0. 0. 0. 0.]
y: [1. 0. 0. 0.]
whole columns at once: [[1. 0. 1. 0. 1. 0.]
 [1. 0. 1. 0. 1. 0.]
 [1. 0. 1. 0. 1. 0.]]


### Exercise 1 — a mask with `.at`

Build an `(8, 8)` array of zeros and ones in which a *column* is either fully on or fully off:
every 4th column (columns 0 and 4) **and** the two central columns (3 and 4) are on; everything else is off.
This is exactly the shape of the k-space masks you'll build in `masks.py`.

In [6]:
mask = jnp.zeros((8, 8))
# YOUR CODE HERE
mask = mask.at[:, ::4].set(1.0)
mask = mask.at[:, 3:5].set(1.0)
mask

Array([[1., 0., 0., 1., 1., 0., 0., 0.],
       [1., 0., 0., 1., 1., 0., 0., 0.],
       [1., 0., 0., 1., 1., 0., 0., 0.],
       [1., 0., 0., 1., 1., 0., 0., 0.],
       [1., 0., 0., 1., 1., 0., 0., 0.],
       [1., 0., 0., 1., 1., 0., 0., 0.],
       [1., 0., 0., 1., 1., 0., 0., 0.],
       [1., 0., 0., 1., 1., 0., 0., 0.]], dtype=float32)

<details><summary><b>Hint</b> — try the exercise first, then click to reveal</summary>

Two `.at[...].set(1.0)` calls, one after the other — remember each returns a *new* array, so
reassign `mask` each time. Section 2 above shows how to address whole columns; a stride-4 column
slice is `[:, ::4]`, and the central band is columns `3:5`.

</details>

In [7]:
# check
assert mask.shape == (8, 8)
assert float(mask.sum()) == 24.0, "3 columns x 8 rows should be on"
assert bool(jnp.all(mask == mask[0][None, :])), "every row must be identical (whole columns on/off)"
print("exercise 1 OK")

exercise 1 OK


## 3. Randomness is explicit

NumPy hides a global random state; JAX makes you pass a **key**. The same key always gives the same
numbers (good: reproducible), so to get *different* numbers you `split` a key into new ones. Rule of
thumb: never reuse a key; split it and hand out the pieces.

In [8]:
key = jax.random.PRNGKey(0)
print(jax.random.normal(key, (3,)))
print(jax.random.normal(key, (3,)), "<- same key, same numbers")

key, sub = jax.random.split(key)        # the idiom: keep `key`, spend `sub`
print(jax.random.normal(sub, (3,)), "<- a fresh subkey, fresh numbers")

[ 1.6226422   2.0252647  -0.43359444]
[ 1.6226422   2.0252647  -0.43359444] <- same key, same numbers
[-2.4424558  -2.0356805   0.20554423] <- a fresh subkey, fresh numbers


### Exercise 2 — two independent draws

From one root key, produce two *different* vectors `a` and `b` of 1000 standard-normal samples.

In [9]:
root = jax.random.PRNGKey(42)
# YOUR CODE HERE
key_a, key_b = jax.random.split(root)
a = jax.random.normal(key_a, (1000,))
b = jax.random.normal(key_b, (1000,))


<details><summary><b>Hint</b> — try the exercise first, then click to reveal</summary>

`jax.random.split(root)` gives two fresh keys; spend one on each draw:
`a = jax.random.normal(ka, (1000,))` and likewise for `b`. If `a` and `b` come out identical,
you reused a key.

</details>

In [10]:
# check
assert a.shape == b.shape == (1000,)
assert not bool(jnp.allclose(a, b)), "a and b must differ - did you reuse a key?"
assert abs(float(a.mean())) < 0.15 and abs(float(b.mean())) < 0.15
assert abs(float(jnp.corrcoef(a, b)[0, 1])) < 0.1, "a and b should be independent"
print("exercise 2 OK")

exercise 2 OK


## 4. `jax.grad` — derivatives for free

Give `jax.grad` a function that returns a scalar and it returns a function that computes the gradient
with respect to the first argument. This is the engine under everything: the VAE is trained with it,
and NumPyro's MAP and NUTS use it to move through the posterior.

In [11]:
def f(x):
    return jnp.sum(x ** 2)         # a scalar

x = jnp.array([1.0, -2.0, 3.0])
print("f(x) =", f(x))
print("grad f(x) =", jax.grad(f)(x), " (expected 2x =", 2 * x, ")")

f(x) = 14.0
grad f(x) = [ 2. -4.  6.]  (expected 2x = [ 2. -4.  6.] )


### Exercise 3 — the gradient of a mean-squared error

Write `mse(x, y) = mean((x - y)^2)` and use `jax.grad` to get its gradient with respect to `x`.
Then verify it against the formula `2 (x - y) / n`.

In [15]:
def mse(x, y):
    # YOUR CODE HERE
    return jnp.mean((x - y)**2)

x = jnp.array([0.5, 1.5, -1.0, 2.0])
y = jnp.array([0.0, 1.0, -2.0, 2.0])
g = jax.grad(mse)(x, y)            # YOUR CODE HERE: the gradient of mse with respect to x, evaluated at (x, y)
g_formula = 2 * (x - y) / x.size    # YOUR CODE HERE: the formula from the statement

<details><summary><b>Hint</b> — try the exercise first, then click to reveal</summary>

`jax.grad(mse)` returns a *function*; call it on the same arguments: `jax.grad(mse)(x, y)`
(the gradient is taken with respect to the first argument). For the formula, `n` is `x.size`.

</details>

In [16]:
# check
assert g is not ... and g_formula is not ..., "fill in the exercise above first"
assert g.shape == x.shape
assert bool(jnp.allclose(g, g_formula, atol=1e-6)), "gradient does not match 2(x - y)/n"
print("exercise 3 OK ·", g)

exercise 3 OK · [0.25 0.25 0.5  0.  ]


## 5. `jit` and `vmap`

`jax.jit` compiles a function the first time you call it (slow once, fast forever after). `jax.vmap`
turns a function written for *one* example into one that works on a *batch*, without you writing a loop.
The repo uses both constantly: `vmap` to score a batch of images, `jit` around every training step.

In [17]:
import time

def slow_norm(x):
    return jnp.sqrt(jnp.sum(x ** 2))

fast_norm = jax.jit(slow_norm)
big = jnp.ones((2000, 2000))
t = time.perf_counter(); fast_norm(big).block_until_ready(); print(f"first call (compiles): {time.perf_counter()-t:.3f}s")
t = time.perf_counter(); fast_norm(big).block_until_ready(); print(f"second call:           {time.perf_counter()-t:.4f}s")

per_image_mean = lambda img: img.mean()             # written for ONE image ...
stack = jnp.arange(5 * 4 * 4, dtype=jnp.float32).reshape(5, 4, 4)
print("vmap over a stack of 5 images:", jax.vmap(per_image_mean)(stack))   # ... applied to five

first call (compiles): 0.047s
second call:           0.0034s
vmap over a stack of 5 images: [ 7.5 23.5 39.5 55.5 71.5]


### Exercise 4 — per-image error with `vmap`

`imgs` and `refs` are stacks of five `16 × 16` images. Using `mse` from exercise 3 and `jax.vmap`,
compute the five per-image errors *without a Python loop*.

In [19]:
key = jax.random.PRNGKey(1)
k1, k2 = jax.random.split(key)
refs = jax.random.uniform(k1, (5, 16, 16))
imgs = refs + 0.1 * jax.random.normal(k2, (5, 16, 16))

errors = jax.vmap(mse)(imgs, refs)     # YOUR CODE HERE
errors

Array([0.01015061, 0.01066093, 0.00971006, 0.01014077, 0.01064433],      dtype=float32)

<details><summary><b>Hint</b> — try the exercise first, then click to reveal</summary>

`jax.vmap` maps a function over the *leading axis* of every argument, so `jax.vmap(mse)` is a
version of `mse` that takes the two stacks directly.

</details>

In [20]:
# check
assert errors is not ..., "fill in the exercise above first"
loop_version = jnp.array([mse(imgs[i], refs[i]) for i in range(5)])
assert errors.shape == (5,)
assert bool(jnp.allclose(errors, loop_version, atol=1e-6))
print("exercise 4 OK")

exercise 4 OK


## 6. Bridge to the repo — your first two milestones

Open `src/mrigen/metrics.py`. Two functions are `TODO`s:

- `psnr(gt, pred, data_range)` = `10 · log10(data_range² / MSE)` — peak signal-to-noise ratio in dB, higher is better;
- `nmse(gt, pred)` = `‖pred − gt‖² / ‖gt‖²` — normalised MSE, lower is better.

They are **deliberately plain NumPy**, even though you just spent an hour on JAX. JAX earns its keep
where something must be differentiated, compiled, or `vmap`ped — the decoder, the FFTs, the
likelihood. Metrics are none of that: they run once per reconstruction, *after* inference, on the
CPU, and feed plain floats into tables and plots; in `jnp` they would ship every image to the
accelerator and back for a subtraction and a log. Real JAX codebases are shaped exactly like this —
a JAX core with NumPy around it — and section 1's `jnp.asarray` / `np.asarray` is how arrays cross
the line. Deciding *where that boundary sits* is as much a JAX skill as `grad` and `vmap`. So inside
`metrics.py`, write `np`, not `jnp` — JAX is not imported there on purpose. (Every repo file states
which library it speaks; check the imports at the top before you write.)

Each function is two lines, and each has a test. Implement them **one at a time**: each has its own check cell below (they reload the module for
you, no kernel restart needed) and its own test you can run in a terminal. While a function still
raises `NotImplementedError` its test reports itself as *skipped*; the moment your implementation is
in, the same command reports *passed* — that flip is your signal. When both are done,
`pixi run milestones` will say `2/8`.

In [23]:

# check psnr on its own
import importlib
import mrigen.metrics
importlib.reload(mrigen.metrics)          # picks up your edit without restarting the kernel
from mrigen.metrics import psnr

gt = np.random.default_rng(0).random((32, 32))
noisy = gt + 0.05 * np.random.default_rng(1).standard_normal((32, 32))
try:
    print(f"PSNR(gt, gt)    = {psnr(gt, gt, data_range=1.0):.1f} dB   (should be huge / inf)")
    print(f"PSNR(gt, noisy) = {psnr(gt, noisy, data_range=1.0):.1f} dB   (about 26 dB)")
    assert psnr(gt, gt, data_range=1.0) > 80
    assert 24 < psnr(gt, noisy, data_range=1.0) < 28
    print("psnr OK - confirm with:  pixi run test tests/test_metrics.py -k psnr")
except NotImplementedError as e:
    print("Not yet:", e)
    print("-> implement psnr in src/mrigen/metrics.py, then re-run this cell")

ERROR: Invalid requirement: 'mrigen#': Expected end or semicolon (after name and no valid version specifier)
    mrigen#
          ^


ModuleNotFoundError: No module named 'mrigen'

In [ ]:
# check nmse on its own
importlib.reload(mrigen.metrics)
from mrigen.metrics import nmse

try:
    print(f"NMSE(gt, noisy) = {nmse(gt, noisy):.4f}      (about 0.007)")
    assert nmse(gt, gt) < 1e-12
    assert 0.004 < nmse(gt, noisy) < 0.012
    print("nmse OK - confirm with:  pixi run test tests/test_metrics.py -k nmse")
except NotImplementedError as e:
    print("Not yet:", e)
    print("-> implement nmse in src/mrigen/metrics.py, then re-run this cell")

## Done when

- all four checks print OK;
- `pixi run milestones` reports `2/8`;
- you can say in one sentence each what `.at[].set()`, `jax.random.split`, `jax.grad` and `jax.vmap` do.

Next: **Prep 2 · Fourier transforms and k-space.**